In [1]:
# Tools
import pandas as pd
from pathlib import Path
import numpy as np

# Preprocessing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.neighbors import KNeighborsClassifier

# Evaluation Metrics
from sklearn.metrics import classification_report, f1_score, accuracy_score
from sklearn.model_selection import learning_curve 

# Guardar modelo
import joblib

In [2]:
CSV_INPUT = Path(r"C:\Users\david\Documents\TFG\train_dataset.csv")

data = pd.read_csv(CSV_INPUT)
data

,video_id,stance_label,effect_label,knee_start,knee_min,knee_target,ankle_drag,ankle_min_dist,hip_tilt,jump,impact_frame,lateral_offset,arm_extension
0,flat_pinpoint_103_landmarks,0,0,168.74,138.30,152.53,25.0840,0.2729,1.0000,6.8047,202,0.5054,167.2977
1,flat_pinpoint_107_landmarks,0,0,160.42,144.11,157.38,2.0281,1.4618,0.6861,8.0920,365,0.6803,167.2809
2,flat_pinpoint_113_landmarks,0,0,162.51,106.18,140.36,28.2208,0.0728,0.5855,4.6137,504,1.3002,167.4573
3,flat_pinpoint_115_landmarks,0,0,174.32,123.50,131.55,6.3603,0.1099,0.8439,3.5063,187,-0.9015,177.0143
4,flat_pinpoint_116_landmarks,0,0,156.98,137.55,140.96,17.1229,0.2096,0.6289,2.3449,444,-3.1157,107.1831
...,...,...,...,...,...,...,...,...,...,...,...,...,...
305,slice_platform_76_landmarks,1,2,173.95,130.06,143.88,4.5566,2.8044,0.4693,3.2438,228,0.8581,168.1534
306,slice_platform_77_landmarks,1,2,164.73,126.63,138.50,1.7986,2.8344,0.5489,8.4448,364,1.0551,172.0871
307,slice_platform_82_landmarks,1,2,161.63,128.45,141.59,3.5252,2.1181,0.4736,8.0483,381,0.6921,162.6849
308,slice_platform_85_landmarks,1,2,179.99,137.06,142.16,7.2953,0.6506,0.6777,4.0673,71,0.7387,161.8273


*We set a configuration specific for each metric*

In [3]:
CONFIGS = {
    "stance": {
        "features": ['knee_start', 'knee_min', 'knee_target', 'ankle_drag', 'ankle_min_dist', 'hip_tilt'],
        "target": "stance_label",
        "mlp_layers": (32, 16),
        "mlp_iter": 2000,
        "mlp_alpha": 0.01,
        "class_weight": None,
    },
    "effect": {
        "features": ['knee_start', 'knee_min', 'knee_target', 'hip_tilt', 'jump', 'impact_frame', 'lateral_offset', 'arm_extension'],
        "target": "effect_label",
        # Configuración para Efecto (quizás necesita más neuronas o distintas iteraciones)
        "mlp_layers": (64, 32), 
        "mlp_iter": 2000,
        "mlp_alpha": 0.001,
        "class_weight": "balanced", # IMPORTANTE: Efecto tiene muy pocos Kicks
    }
}

*Creamos una función para hallar el modelo ganador*

In [5]:
def get_model(df, config_name):
    cfg = CONFIGS[config_name]
    print(f"TRAINING: {config_name.upper()}")
    
    X = df[cfg["features"]]
    y = df[cfg["target"]]
    
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=42
    )

    print(f"Data divided")
    print(f"    -   We train with {len(X_train)} videos")
    print(f"    -   We test with {len(X_test)} videos")
    
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    modelos = {
    "Logistic Regression": LogisticRegression(class_weight=cfg["class_weight"], random_state=42),
    "SVM (Support Vector)": SVC(kernel='rbf', class_weight=cfg["class_weight"], probability=True, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, class_weight=cfg["class_weight"], random_state=42),
    "Red Neuronal (MLP)": MLPClassifier(
        hidden_layer_sizes=cfg["mlp_layers"], 
        max_iter=cfg["mlp_iter"], 
        alpha=cfg["mlp_alpha"], 
        random_state=42
    ),
    "KNN (Vecinos)": KNeighborsClassifier(n_neighbors=5)
    }
    
    results = []
    
    print("Iniciando entrenamiento y evaluación\n")

    for nombre, modelo in modelos.items():
        modelo.fit(X_train_scaled, y_train)
        y_pred = modelo.predict(X_test_scaled)
        
        f1 = f1_score(y_test, y_pred, average='weighted')
        acc = accuracy_score(y_test, y_pred)
        
        results.append({
            "Modelo": nombre,
            "F1-Score": f1,
            "Accuracy": acc
        })
        
    
    df_ranking = pd.DataFrame(results)
    df_ranking = df_ranking.sort_values(by="F1-Score", ascending=False)
    
    print(f"RANKING PARA {config_name.upper()}:")
    print(df_ranking.to_string(index=False)+"\n")


In [4]:
import joblib # No olvides importar esto arriba del todo

def get_model(df, config_name):
    cfg = CONFIGS[config_name]
    print(f"TRAINING: {config_name.upper()}")
    
    X = df[cfg["features"]]
    y = df[cfg["target"]]
    
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=42
    )

    print(f"Data divided")
    print(f"    -   We train with {len(X_train)} videos")
    print(f"    -   We test with {len(X_test)} videos")
    
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    modelos = {
        "Logistic Regression": LogisticRegression(class_weight=cfg["class_weight"], random_state=42),
        "SVM (Support Vector)": SVC(kernel='rbf', class_weight=cfg["class_weight"], probability=True, random_state=42),
        "Random Forest": RandomForestClassifier(n_estimators=100, class_weight=cfg["class_weight"], random_state=42),
        "Red Neuronal (MLP)": MLPClassifier(
            hidden_layer_sizes=cfg["mlp_layers"], 
            max_iter=cfg["mlp_iter"], 
            alpha=cfg["mlp_alpha"], 
            random_state=42
        ),
        "KNN (Vecinos)": KNeighborsClassifier(n_neighbors=5)
    }
    
    results = []
    
    # --- VARIABLES PARA GUARDAR EL GANADOR ---
    best_f1 = 0
    best_model = None
    
    print("Iniciando entrenamiento y evaluación\n")

    for nombre, modelo in modelos.items():
        modelo.fit(X_train_scaled, y_train)
        y_pred = modelo.predict(X_test_scaled)
        
        f1 = f1_score(y_test, y_pred, average='weighted')
        acc = accuracy_score(y_test, y_pred)
        
        results.append({
            "Modelo": nombre,
            "F1-Score": f1,
            "Accuracy": acc
        })
        
        # --- SI SUPERA EL RÉCORD, LO GUARDAMOS EN LA VARIABLE ---
        if f1 > best_f1:
            best_f1 = f1
            best_model = modelo
            
    
    df_ranking = pd.DataFrame(results)
    df_ranking = df_ranking.sort_values(by="F1-Score", ascending=False)
    
    print(f"RANKING PARA {config_name.upper()}:")
    print(df_ranking.to_string(index=False)+"\n")

    # ==========================================
    # GUARDAR EL MEJOR MODELO Y SU SCALER
    # ==========================================
    nombre_modelo = f"model_{config_name}.pkl"
    nombre_scaler = f"scaler_{config_name}.pkl"
    
    joblib.dump(best_model, nombre_modelo)
    joblib.dump(scaler, nombre_scaler)
    
    # Extraemos el nombre del modelo ganador para imprimirlo
    ganador_nombre = df_ranking.iloc[0]["Modelo"]
    
    print(f"💾 ¡Archivos guardados con éxito!")
    print(f"   -> Cerebro ({ganador_nombre}): {nombre_modelo}")
    print(f"   -> Escalador: {nombre_scaler}\n")
    print("-" * 50)

In [5]:
get_model(data, 'stance')
get_model(data, 'effect')

TRAINING: STANCE
Data divided
    -   We train with 248 videos
    -   We test with 62 videos
Iniciando entrenamiento y evaluación

RANKING PARA STANCE:
              Modelo  F1-Score  Accuracy
  Red Neuronal (MLP)  0.870968  0.870968
       Random Forest  0.809553  0.806452
 Logistic Regression  0.808963  0.806452
       KNN (Vecinos)  0.793833  0.790323
SVM (Support Vector)  0.793803  0.790323

💾 ¡Archivos guardados con éxito!
   -> Cerebro (Red Neuronal (MLP)): model_stance.pkl
   -> Escalador: scaler_stance.pkl

--------------------------------------------------
TRAINING: EFFECT
Data divided
    -   We train with 248 videos
    -   We test with 62 videos
Iniciando entrenamiento y evaluación

RANKING PARA EFFECT:
              Modelo  F1-Score  Accuracy
       KNN (Vecinos)  0.545881  0.612903
       Random Forest  0.519654  0.580645
  Red Neuronal (MLP)  0.481300  0.483871
 Logistic Regression  0.479854  0.467742
SVM (Support Vector)  0.478330  0.467742

💾 ¡Archivos guardados con é